In [1]:
%load_ext autoreload
%autoreload 2

from PIL import Image
import os
import pathlib 

import png_background_converter as helper

In [2]:
folder_path = "./new"
# white to transparent
images = ["Turtle.png", "tree.png", "herons_colors_vertical.png", "front_turtle.png"]
images = ["Triskel.png"]

# Remove Background

In [3]:
image_path = "./dom_sonne.png"

# remove white background (keeping transparent same)
# helper.remove_white_background(image_path)

# remove black background (keeping transparent same)
#helper.remove_black_background("./herons_white.png")

In [6]:
def prep_white_on_black_motif(to_process_folder="./to_process", transparent_folder= "./transparent", white_folder =  "./white_transparent"):
    pathlib.Path(white_folder).mkdir(exist_ok=True)
    pathlib.Path(transparent_folder).mkdir(exist_ok=True)
    
    exts = {".jpg", ".jpeg", ".png"}
    for p in sorted(pathlib.Path(to_process_folder).iterdir()):
        if p.is_file() and p.suffix.lower() in exts:    
            print(f"File {p}")
            # remove white background
            transparent_path = helper.remove_white_background(p, result_dir=transparent_folder)
            # convert black to white pixels
            transparent_path = helper.black_to_white(transparent_path,  result_dir=white_folder)
    
prep_white_on_black_motif(to_process_folder="test")

File test\home_dome.png
Background transparent saved: transparent\home_dome_transparent.png
Black to white saved: ./white_transparent/home_dome_transparent_white.png
File test\tree.png
Background transparent saved: transparent\tree_transparent.png
Black to white saved: ./white_transparent/tree_transparent_white.png


In [15]:
# black to transparent
black_remove_images = ["herons_colors_vertical.png"]
for i in black_remove_images:
    remove_black_background(pathlib.Path(folder_path, i))

file_name: herons_colors_vertical_transparent.png
Saved: new/herons_colors_vertical_transparent.png


# SVG processing

In [36]:
import re
import xml.etree.ElementTree as ET
from pathlib import Path


def add_svg_padding(svg_path, padding=10, output_path=None, relative=False):
    """
    Add transparent padding on all 4 sides of an SVG by enlarging its viewBox.

    padding:  padding in SVG user units (the viewBox coordinate system),
              or a fraction of the longer side if relative=True (e.g. 0.05 = 5%).
    output_path: where to save; overwrites the input file if None.
    """
    svg_path = Path(svg_path)
    output_path = Path(output_path) if output_path else svg_path

    # Keep the original namespace prefixes (otherwise they become ns0:, ns1: ...)
    for _, (prefix, uri) in ET.iterparse(svg_path, events=["start-ns"]):
        ET.register_namespace(prefix, uri)

    tree = ET.parse(svg_path)
    root = tree.getroot()

    def split_unit(value):
        m = re.fullmatch(r"\s*([\d.]+)\s*([a-z%]*)\s*", value or "")
        return (float(m.group(1)), m.group(2)) if m else (None, "")

    width, w_unit = split_unit(root.get("width"))
    height, h_unit = split_unit(root.get("height"))

    # Get the current viewBox, or derive one from width/height
    vb = root.get("viewBox")
    if vb:
        x, y, w, h = map(float, re.split(r"[,\s]+", vb.strip()))
    elif width and height:
        x, y, w, h = 0.0, 0.0, width, height
    else:
        raise ValueError("SVG has neither a viewBox nor numeric width/height.")

    pad = padding * max(w, h) if relative else padding
    new_w, new_h = w + 2 * pad, h + 2 * pad
    root.set("viewBox", f"{x - pad:g} {y - pad:g} {new_w:g} {new_h:g}")

    # Scale width/height so the artwork keeps its original size
    # (the image just gets bigger by the padding, not zoomed out)
    if width and w_unit != "%":
        root.set("width", f"{width * new_w / w:g}{w_unit}")
    if height and h_unit != "%":
        root.set("height", f"{height * new_h / h:g}{h_unit}")

    tree.write(output_path, encoding="utf-8", xml_declaration=True)
    return output_path


# Examples
add_svg_padding("Tree.svg", padding=50, output_path="tree_padded.svg")
# add_svg_padding("tree.svg", padding=0.05, relative=True)  # 5% of longer side

# add_svg_padding()

WindowsPath('tree_padded.svg')

In [2]:
import re

def make_svg_white(input_path: str, output_path: str) -> None:
    with open(input_path, "r", encoding="utf-8") as file:
        svg = file.read()

    # Replace fill and stroke attributes
    svg = re.sub(
        r'((?:fill|stroke)\s*=\s*["\'])[^"\']*(["\'])',
        lambda m: m.group(1) + "white" + m.group(2),
        svg,
        flags=re.IGNORECASE,
    )

    # Replace colors in inline styles
    svg = re.sub(
        r'((?:fill|stroke)\s*:\s*)[^;}"\']+',
        lambda m: m.group(1) + "white",
        svg,
        flags=re.IGNORECASE,
    )

    with open(output_path, "w", encoding="utf-8") as file:
        file.write(svg)

In [3]:
make_svg_white("tree_padded.svg", "tree_padded_white.svg")

# TEST: Remove page structure from scan


Zum Nachjustieren: wenn noch Körnung übrig bleibt, t_weak senken (0.92) oder min_area erhöhen. Wenn Striche verschwinden, t_weak anheben (0.96) — durch die Hysterese ist das relativ gutmütig. bg_radius muss größer sein als deine dickste Linie, sonst frisst das Closing sie als „Hintergrund" auf.

In [14]:
"""
Papierstruktur (Korn / Faserstruktur) aus gescannten Tuschezeichnungen entfernen,
ohne gezeichnete Linien anzutasten.
"""
import cv2
import numpy as np
from scipy import ndimage
import os

def clean_scan(
    path,
    bg_radius=25,
    t_strong=0.80,
    t_weak=0.94,
    min_area=6,
    keep_gray=True,
):
    """
    Papierstruktur aus einem gescannten Bild entfernen, ohne Striche anzutasten.

    path      : Pfad zur Eingangsdatei (jpg, png, tif ...)
    bg_radius : Radius der Hintergrundschaetzung, muss groesser sein als die
                dickste Strichbreite.
    t_strong  : unter diesem Anteil der lokalen Papierhelligkeit = sicher Tusche
    t_weak    : darueber = sicher Papier; dazwischen nur behalten, wenn es an
                sichere Tusche angrenzt
    min_area  : isolierte Flecken kleiner als das (in Pixeln) fliegen raus
    keep_gray : True = Linien behalten ihre Graustufen, False = reines S/W

    Rueckgabe : Graustufenbild als uint8-Array, Hintergrund exakt 255.
    """
    gray = cv2.imdecode(np.fromfile(path, np.uint8), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f"Konnte Bild nicht laden: {path}")
    g = gray.astype(np.float32)

    # 1) Hintergrund schaetzen: Closing schliesst alle Striche, uebrig bleibt
    #    der Papierton inkl. Beleuchtungsverlauf des Scanners.
    k = 2 * bg_radius + 1
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    bg = cv2.morphologyEx(g, cv2.MORPH_CLOSE, se)
    bg = cv2.GaussianBlur(bg, (0, 0), bg_radius / 2.0)
    bg = np.maximum(bg, 1.0)

    # 2) Normalisieren: 1.0 = Papier. Schwellwerte werden damit unabhaengig
    #    von Scannerhelligkeit und Vignette.
    norm = np.clip(g / bg, 0.0, 1.0)

    # 3) Hysterese wie bei Canny: unsichere Pixel nur behalten, wenn sie mit
    #    sicherer Tusche zusammenhaengen. Rettet blasse Schraffur, killt Korn.
    strong = norm < t_strong
    weak = norm < t_weak

    lbl, n = ndimage.label(weak, structure=np.ones((3, 3), np.uint8))
    if n:
        keep_ids = np.unique(lbl[strong])
        keep_ids = keep_ids[keep_ids != 0]
        ink = np.isin(lbl, keep_ids)
    else:
        ink = weak

    # 4) Restliche Einzelflecken (Staub, Scannerrauschen) entfernen.
    if min_area > 1:
        lbl2, n2 = ndimage.label(ink, structure=np.ones((3, 3), np.uint8))
        if n2:
            areas = np.bincount(lbl2.ravel())
            ink = ink & (areas[lbl2] >= min_area)

    # 5) Zusammensetzen: Hintergrund exakt weiss, Tusche unveraendert.
    out = np.full(gray.shape, 255, np.uint8)
    if keep_gray:
        stretched = np.clip(norm / t_weak, 0.0, 1.0) * 255.0
        out[ink] = stretched[ink].astype(np.uint8)
    else:
        out[ink] = 0
        
    base, _ = os.path.splitext(path) 
    out_path = f"{base}_clean.png"

    ext = "." + str(out_path).rsplit(".", 1)[-1]
    ok, buf = cv2.imencode(ext, out)
    if not ok:
        raise ValueError(f"Konnte Format nicht schreiben: {ext}")
    buf.tofile(out_path)


In [19]:
"""
Gescannte Tuschezeichnung -> PNG mit transparentem Hintergrund.
Die Deckkraft (Alpha) kommt direkt aus der Dunkelheit des Strichs.
"""

import cv2
import numpy as np
from scipy import ndimage

_S = np.ones((3, 3), np.uint8)


def _area_filter(mask, min_area):
    """Zusammenhaengende Flecken unterhalb min_area entfernen."""
    if min_area <= 1:
        return mask
    lbl, n = ndimage.label(mask, structure=_S)
    if not n:
        return mask
    areas = np.bincount(lbl.ravel())
    return mask & (areas[lbl] >= min_area)


def scan_to_rgba(
    path,
    bg_radius=25,
    t_strong=0.80,
    t_weak=0.94,
    seed_min_area=8,
    min_area=12,
    gamma=1.0,
    ink_color=(0, 0, 0),
    keep_color=False,
    trim=0,
    white_background=False,
):
    """
    path          : Pfad zur Eingangsdatei
    bg_radius     : Radius der Hintergrundschaetzung, groesser als die dickste Linie
    t_strong      : sicher Tusche (Anteil der lokalen Papierhelligkeit)
    t_weak        : sicher Papier; dazwischen nur behalten bei Anschluss an Tusche
    seed_min_area : Mindestgroesse, damit ein dunkler Fleck als Strich-Saat zaehlt.
                    Das ist der Regler gegen dunkle Papierkoerner.
    min_area      : Mindestgroesse der fertigen Strich-Fragmente
    gamma         : >1 macht blasse Striche transparenter, <1 deckender
    ink_color     : Strichfarbe als (B, G, R)
    keep_color    : True -> Originalfarbe der Striche behalten
    trim          : Pixel-Rand, der komplett transparent gesetzt wird (Blattkante)

    Rueckgabe     : BGRA-Array, uint8
    """
    color = cv2.imdecode(np.fromfile(path, np.uint8), cv2.IMREAD_COLOR)
    if color is None:
        raise FileNotFoundError(f"Konnte Bild nicht laden: {path}")
    gray = cv2.cvtColor(color, cv2.COLOR_BGR2GRAY)
    g = gray.astype(np.float32)

    # 1) Papierton lokal schaetzen und rausrechnen
    k = 2 * bg_radius + 1
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    bg = cv2.morphologyEx(g, cv2.MORPH_CLOSE, se)
    bg = cv2.GaussianBlur(bg, (0, 0), bg_radius / 2.0)
    bg = np.maximum(bg, 1.0)
    norm = np.clip(g / bg, 0.0, 1.0)          # 1.0 = Papier, 0.0 = volle Tusche

    # 2) Alpha = Deckung, kontinuierlich. Kein harter Schnitt -> weiche Kanten
    #    bleiben weich, statt treppig zu werden.
    alpha = np.clip((t_weak - norm) / t_weak, 0.0, 1.0)
    if gamma != 1.0:
        alpha = alpha ** gamma

    # 3) Maske per Hysterese. Die Saat wird vorher entkoernt, damit einzelne
    #    dunkle Papierkoerner keine Strichfragmente ins Leben rufen.
    strong = _area_filter(norm < t_strong, seed_min_area)
    weak = norm < t_weak
    lbl, n = ndimage.label(weak, structure=_S)
    if n:
        keep_ids = np.unique(lbl[strong])
        keep_ids = keep_ids[keep_ids != 0]
        ink = np.isin(lbl, keep_ids)
    else:
        ink = weak
    ink = _area_filter(ink, min_area)

    # 4) Maske als weiches Tor aufs Alpha legen: Korn faellt auf 0, die Striche
    #    behalten ihren Alphaverlauf.
    gate = cv2.GaussianBlur(ink.astype(np.float32), (0, 0), 0.6)
    alpha = alpha * np.clip(gate * 1.4, 0.0, 1.0)

 # 5) Blattkante wegschneiden
    if trim > 0:
        alpha[:trim, :] = 0
        alpha[-trim:, :] = 0
        alpha[:, :trim] = 0
        alpha[:, -trim:] = 0

    # 6) RGBA bauen. RGB ist ueberall gesetzt, auch unter transparenten Pixeln,
    #    sonst zeigen manche Programme schwarze Saeume an den Kanten.
    a8 = (alpha * 255).round().astype(np.uint8)
    if keep_color:
        rgb = color
    else:
        rgb = np.zeros_like(color)
        rgb[:] = np.array(ink_color, np.uint8)

    if white_background:
        # Striche mit ihrem Alpha ueber Weiss legen (Alpha-Compositing).
        # Nicht einfach hart einsetzen - sonst verlieren die weichen
        # Federkanten genau die Halbtoene, die sie weich machen.
        a = alpha[..., None]
        out = (rgb.astype(np.float32) * a + 255.0 * (1.0 - a))
        out = out.round().astype(np.uint8)
    else:
        out = np.dstack([rgb, a8])

    base, _ = os.path.splitext(path) 
    out_path = f"{base}_clean.png"

    ext = "." + str(out_path).rsplit(".", 1)[-1]
    ok, buf = cv2.imencode(".png", out)
    if not ok:
        raise ValueError(f"Konnte Format nicht schreiben: {ext}")
    buf.tofile(out_path)


In [20]:

# Example usage
# remove_background_and_darken("eyegenart.jpg", threshold=245)

#clean_scan("sketch_test.jpg",t_weak=0.9,min_area=6)

In [22]:
scan_to_rgba("sketch_test.jpg", white_background=True, trim=10)

# TEST: Correct uneven exposure (flash hotspot / side lighting)

Estimates a smooth illumination map (same background-estimation trick as `clean_scan`
above: heavy blur, then divide it out) and flattens it. Works on the Lab L channel only,
so color and saturation of the artwork stay untouched — only brightness is corrected.

In [9]:
import cv2
import numpy as np
import pathlib


def estimate_illumination_map(channel, blur_sigma, downscale=8):
    """Estimate a smooth illumination field for a single-channel image.
    Blurring is done on a downscaled copy (much faster, and fine detail gets
    smoothed away anyway) then the result is upscaled back to full size.
    Bright regions of the returned map are the areas that were more exposed."""
    h, w = channel.shape
    small = cv2.resize(
        channel, (max(1, w // downscale), max(1, h // downscale)),
        interpolation=cv2.INTER_AREA,
    )
    small_sigma = max(blur_sigma / downscale, 1.0)
    blurred = cv2.GaussianBlur(small.astype(np.float32), (0, 0), small_sigma)
    return cv2.resize(blurred, (w, h), interpolation=cv2.INTER_CUBIC)


def apply_illumination_correction(channel, illumination_map, target=None):
    """Divide channel by illumination_map to flatten exposure, then rescale
    onto `target` (default: the map's mean) so overall brightness is kept
    instead of the whole image going flat and dark."""
    illumination_map = np.maximum(illumination_map, 1.0)
    if target is None:
        target = float(illumination_map.mean())
    corrected = channel.astype(np.float32) / illumination_map * target
    return np.clip(corrected, 0, 255).astype(np.uint8)


def correct_exposure(image_path, blur_sigma=None, downscale=8, result_dir=None):
    """Fix uneven exposure (flash hotspot, light falling off to one side) in a
    photo of artwork. Only the Lab L channel (lightness) is corrected, so hue
    and saturation are left alone.

    blur_sigma : radius of the estimated illumination field. None = auto
                 (a quarter of the shorter image edge). Must be larger than
                 the biggest details in the artwork, or those get smoothed
                 away as if they were part of the lighting.
    downscale  : how much to shrink the image before estimating illumination
                 (speed only, no real quality loss since it's heavily blurred
                 anyway).

    Note: a blown-out flash hotspot (clipped to pure white) has no recoverable
    detail — this can even out the surrounding gradient but not un-clip it.
    """
    img = cv2.imdecode(np.fromfile(image_path, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")

    h, w = img.shape[:2]
    if blur_sigma is None:
        blur_sigma = min(h, w) / 4.0

    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    illumination_map = estimate_illumination_map(l_channel, blur_sigma, downscale)
    corrected_l = apply_illumination_correction(l_channel, illumination_map)

    corrected_lab = cv2.merge([corrected_l, a_channel, b_channel])
    result = cv2.cvtColor(corrected_lab, cv2.COLOR_LAB2BGR)

    path = pathlib.Path(image_path)
    file_name = f"{path.stem}_exposure_corrected{path.suffix}"
    if not result_dir:
        result_dir = path.parent
    output_path = f"{result_dir}/{file_name}"

    ok, buf = cv2.imencode(path.suffix, result)
    if not ok:
        raise ValueError(f"Could not encode format: {path.suffix}")
    buf.tofile(output_path)
    print(f"Saved: {output_path}")


In [10]:

# Example usage
# correct_exposure("uneven_light.jpg")

# If the auto blur_sigma smooths away real brush strokes / paper texture,
# raise it; if the hotspot isn't fully evened out, lower it.
correct_exposure("one_pad_thai_please.jpg", blur_sigma=400)

Saved: ./one_pad_thai_please_exposure_corrected.jpg
